# Project 1 — Celebrity Detection
## Milestone 2: Synthetic Multi-Celebrity Grid Image Construction

**IE 7615** — Project 1, Milestone 2

**David Fung**

Milestone 2 builds the **synthetic multi-celebrity grid dataset** that Milestone 3 trains YOLOv8 on. Each composite image is a 3×3 grid of celebrity face crops (9 distinct identities per frame) on a varied background, annotated with tight YOLO-format bounding boxes. A detection-appropriate augmentation strategy is documented, and a clean 70/15/15 train/val/test split is produced.

The grid layout guarantees that every face is well-illuminated, unoccluded, and at a predictable scale — making the detection task tractable while still requiring the model to localise multiple identities in a single frame. This is the bridge from single-face classification (Milestone 1) to multi-object detection (Milestone 3).

## Design Decisions

| Parameter | Choice | Rationale |
|---|---|---|
| Canvas size | 640×640 | Matches YOLOv8 `imgsz=640` used in Milestone 3. |
| Grid layout | 3×3 (9 cells) | Nine faces per image; each cell ≈213×213 px. |
| Identities per image | 9 distinct (from 13) | No identity repeats within a frame. |
| Image reuse | No file appears in >1 grid | Maximises diversity. |
| Backgrounds | 50% gradient / 30% noise / 20% blurred face | Prevents background shortcut learning. |
| Face scale in cell | 80–95% of cell size | Fills most of cell; small margin prevents clipping. |
| Face offset | ±5% random jitter | Breaks perfectly-regular grid pattern. |
| Horizontal flip | 50% chance per face | Pose diversity. |
| Brightness jitter | 0.85–1.15 | Mild lighting variation. |
| Split | 70 / 15 / 15 | Standard; identity-balanced. |
| Random seed | 42 | Fully reproducible. |
| Minimum images | 2 (requirement); default 20 | Enough for a meaningful split. |

In [ ]:
import os
import time
import random
import shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageEnhance, ImageFilter
from sklearn.model_selection import train_test_split
import tensorflow as tf

# Deterministic seeding (Milestone 1 convention)
GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
tf.random.set_seed(GLOBAL_SEED)
print(f"TensorFlow: {tf.__version__}")

In [ ]:
# PROJECT CONSTANTS

def set_all_seeds():
    random.seed(GLOBAL_SEED)
    np.random.seed(GLOBAL_SEED)
    tf.random.set_seed(GLOBAL_SEED)

IMG_W, IMG_H = 640, 640
GRID_N = 3
FACES_PER_IMAGE = GRID_N ** 2
NUM_IMAGES = 20
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15

def resolve_project_root():
    cur = os.getcwd()
    for _ in range(8):
        if os.path.isdir(os.path.join(cur, 'data', 'class_images')):
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    raise FileNotFoundError(f"Could not find data/class_images from cwd={os.getcwd()}")

ROOT = resolve_project_root()
FACE_POOL_DIR = os.path.join(ROOT, 'data', 'class_images')
OUTPUT_DIR = os.path.join(ROOT, 'data', 'synthetic_grid')
print(f"Project root : {ROOT}")
print(f"Face pool    : {FACE_POOL_DIR}")
print(f"Output dir   : {OUTPUT_DIR}")

### Task 1 — Load the Face Pool

Thirteen celebrity identities are stored in `data/class_images/<identity_id>/`, each containing 20+ face crops. This cell discovers the identities and builds the identity-to-face-path mapping. All 13 identities are available; 9 are randomly selected per grid image without replacement.

In [ ]:
identity_pools = {}
for name in sorted(os.listdir(FACE_POOL_DIR)):
    class_dir = os.path.join(FACE_POOL_DIR, name)
    if not os.path.isdir(class_dir):
        continue
    paths = sorted(
        os.path.join(class_dir, f)
        for f in os.listdir(class_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    )
    if paths:
        identity_pools[name] = paths

CLASS_NAMES = sorted(identity_pools.keys(), key=lambda x: int(x) if x.isdigit() else x)
name_to_idx = {name: i for i, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)
total_faces = sum(len(v) for v in identity_pools.values())

print(f"Loaded face pool ({total_faces} faces, {NUM_CLASSES} identities):")
for name in CLASS_NAMES:
    print(f"  class {name_to_idx[name]:2d}  ID {name:<6s}  {len(identity_pools[name]):>3d} faces")

assert NUM_CLASSES >= FACES_PER_IMAGE, f"Need >= {FACES_PER_IMAGE} identities, found {NUM_CLASSES}"

### Task 2 — Synthetic Background Generation

Each grid composite is placed on a varied background to prevent the detector from learning a background shortcut. Three families are used:

- **Linear gradient (50%)** — two random muted colours blended top-to-bottom.
- **Blurred noise field (30%)** — Gaussian-blurred random noise, mimicking an out-of-focus scene.
- **Blurred face (20%)** — a random face heavily blurred and darkened so it reads as soft texture, not a second face.

All three keep the foreground celebrity faces as the only sharp objects in the frame.

In [ ]:
def linear_gradient(w, h, c1, c2):
    t = np.linspace(0.0, 1.0, h)[:, None]
    t = np.broadcast_to(t, (h, w))
    c1 = np.asarray(c1, float).reshape(1, 1, 3)
    c2 = np.asarray(c2, float).reshape(1, 1, 3)
    arr = c1 * (1.0 - t[..., None]) + c2 * t[..., None]
    return Image.fromarray(arr.astype(np.uint8), 'RGB')

def bg_gradient(w, h):
    c1 = [random.randint(25, 130) for _ in range(3)]
    c2 = [random.randint(25, 130) for _ in range(3)]
    return linear_gradient(w, h, c1, c2)

def bg_noise(w, h):
    arr = np.random.randint(50, 160, (h, w, 3), dtype=np.uint8)
    return Image.fromarray(arr, 'RGB').filter(ImageFilter.GaussianBlur(radius=random.randint(20, 60)))

def bg_blur_face(w, h):
    all_paths = [p for paths in identity_pools.values() for p in paths]
    if all_paths:
        src = Image.open(random.choice(all_paths)).convert('RGB')
        scale = max(w / src.width, h / src.height) * 1.2
        nw, nh = int(src.width * scale), int(src.height * scale)
        cover = src.resize((nw, nh))
        left, top = (nw - w) // 2, (nh - h) // 2
        img = cover.crop((left, top, left + w, top + h))
        img = img.filter(ImageFilter.GaussianBlur(radius=random.randint(35, 80)))
        return ImageEnhance.Brightness(img).enhance(random.uniform(0.4, 0.7))
    return bg_gradient(w, h)

def generate_background(w, h):
    r = random.random()
    if r < 0.50:
        return bg_gradient(w, h)
    if r < 0.80:
        return bg_noise(w, h)
    return bg_blur_face(w, h)

# Preview
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, fn, title in zip(ax, [bg_gradient, bg_noise, bg_blur_face],
                        ['gradient', 'blurred noise', 'blurred face']):
    a.imshow(fn(IMG_W, IMG_H)); a.axis('off'); a.set_title(title)
plt.suptitle('Task 2 — background families')
plt.tight_layout(); plt.show()

### Task 3 — Grid Compositing Algorithm

Each 640×640 canvas is divided into a 3×3 grid (≈213×213 px per cell). For every grid image:

1. **Identity selection** — 9 of the 13 available identities are chosen at random (without replacement), so no celebrity appears twice in the same frame.
2. **Image selection** — one random face crop is drawn from each chosen identity’s pool. A global `used_paths` set guarantees that no image file is ever reused across the entire dataset.
3. **Per-face augmentation** — 50% horizontal flip, brightness jitter (0.85–1.15), and a random scale of 80–95% of cell size (aspect ratio preserved via centre crop).
4. **Placement** — each face is pasted into its assigned cell with a ±5% random offset to break the perfectly-regular pattern.
5. **Annotation** — a tight YOLO bounding box is recorded for every face: `<class> <cx> <cy> <w> <h>` (normalised to 0–1).

The identity-to-cell assignment is shuffled per image, so the same 9 celebrities appear in different positions across the dataset.

In [ ]:
def prepare_face(img_path, cell_w, cell_h, used_paths):
    """Load, augment, and resize a face crop to fit within a grid cell."""
    if img_path in used_paths:
        return None
    used_paths.add(img_path)
    face = Image.open(img_path).convert('RGB')
    # 50% horizontal flip
    if random.random() < 0.5:
        face = face.transpose(Image.FLIP_LEFT_RIGHT)
    # Brightness jitter
    face = ImageEnhance.Brightness(face).enhance(random.uniform(0.85, 1.15))
    # Scale to 80-95% of cell, preserving aspect ratio via centre crop
    scale = random.uniform(0.80, 0.95)
    target_w = int(cell_w * scale)
    target_h = int(cell_h * scale)
    # Centre-crop to target aspect, then resize
    src_ar = face.width / face.height
    tgt_ar = target_w / target_h
    if src_ar > tgt_ar:
        new_w = int(face.height * tgt_ar)
        left = (face.width - new_w) // 2
        face = face.crop((left, 0, left + new_w, face.height))
    else:
        new_h = int(face.width / tgt_ar)
        top = (face.height - new_h) // 2
        face = face.crop((0, top, face.width, top + new_h))
    face = face.resize((target_w, target_h), Image.BICUBIC)
    return face


def compose_grid(used_paths):
    """Build one 3x3 grid composite image with 9 distinct identities."""
    # Step 1 — select 9 distinct identities
    all_ids = list(identity_pools.keys())
    chosen_ids = random.sample(all_ids, FACES_PER_IMAGE)
    # Shuffle cell assignment so same identities land in different positions
    cell_order = list(range(FACES_PER_IMAGE))
    random.shuffle(cell_order)
    # Step 2 — generate background
    canvas = generate_background(IMG_W, IMG_H)
    # Step 3 — place faces into cells
    cell_w = IMG_W // GRID_N
    cell_h = IMG_H // GRID_N
    yolo_boxes = []
    for cell_idx, ident in enumerate(chosen_ids):
        row, col = divmod(cell_idx, GRID_N)
        # Pick an unused image from this identity
        pool = identity_pools[ident]
        available = [p for p in pool if p not in used_paths]
        if not available:
            # Fallback: allow reuse if pool is exhausted (should not happen with 20+ images)
            available = pool
        img_path = random.choice(available)
        face = prepare_face(img_path, cell_w, cell_h, used_paths)
        if face is None:
            continue
        # Centre in cell with ±5% jitter
        fx = (face.width) // 2
        fy = (face.height) // 2
        jitter_x = random.randint(-cell_w // 20, cell_w // 20)
        jitter_y = random.randint(-cell_h // 20, cell_h // 20)
        paste_x = col * cell_w + (cell_w - face.width) // 2 + jitter_x
        paste_y = row * cell_h + (cell_h - face.height) // 2 + jitter_y
        # Clamp to cell bounds
        paste_x = max(col * cell_w, min(paste_x, (col + 1) * cell_w - face.width))
        paste_y = max(row * cell_h, min(paste_y, (row + 1) * cell_h - face.height))
        canvas.paste(face, (paste_x, paste_y))
        # YOLO box (normalised)
        cx = (paste_x + face.width / 2.0) / IMG_W
        cy = (paste_y + face.height / 2.0) / IMG_H
        bw = face.width / IMG_W
        bh = face.height / IMG_H
        cls_idx = name_to_idx[ident]
        yolo_boxes.append((cls_idx, cx, cy, bw, bh))
    return canvas, yolo_boxes, chosen_ids

### Task 4 — Generate the Synthetic Dataset

Generates `NUM_IMAGES` grid composites (default 20; minimum 2 per the milestone requirement). Each image is written as a 640×640 JPG alongside a YOLO-format label file. A global `used_paths` set ensures no source image is ever reused across the dataset.

In [ ]:
set_all_seeds()

    # Clear the output directory (ignore locked files on Windows)
    if os.path.isdir(OUTPUT_DIR):
        shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Cleared {OUTPUT_DIR} — starting fresh")

# Create staging area
staging = os.path.join(OUTPUT_DIR, 'staging')
os.makedirs(os.path.join(staging, 'images'), exist_ok=True)
os.makedirs(os.path.join(staging, 'labels'), exist_ok=True)

# Track used images globally (no reuse across the dataset)
used_paths = set()
manifest = []

t0 = time.time()
for img_idx in range(NUM_IMAGES):
    canvas, yolo_boxes, chosen_ids = compose_grid(used_paths)
    stem = f"grid_{img_idx:04d}"
    img_path = os.path.join(staging, 'images', stem + '.jpg')
    lbl_path = os.path.join(staging, 'labels', stem + '.txt')
    canvas.save(img_path, quality=90)
    with open(lbl_path, 'w') as f:
        for (ci, cx, cy, bw, bh) in yolo_boxes:
            f.write(f"{ci} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")
    manifest.append({
        'index': img_idx,
        'image': img_path,
        'label': lbl_path,
        'n_faces': len(yolo_boxes),
        'classes': [CLASS_NAMES[b[0]] for b in yolo_boxes],
    })
    if (img_idx + 1) % 5 == 0:
        print(f"  generated {img_idx + 1} / {NUM_IMAGES}")

elapsed = time.time() - t0
print(f"Generated {len(manifest)} grid images → {staging}")
print(f"Time: {elapsed:.1f}s")

# Inline preview: first 2 grid composites
n_preview = min(2, len(manifest))
fig, axes = plt.subplots(1, n_preview, figsize=(7 * n_preview, 7))
for ax, m in zip(np.atleast_1d(axes).flatten(), manifest[:n_preview]):
    ax.imshow(Image.open(m['image']))
    ax.set_title(f"Grid {m['index']} \u2014 {m['n_faces']} faces", fontsize=9)
    ax.axis('off')
plt.suptitle("Generated grid composites (raw)", fontsize=12)
plt.tight_layout()
plt.show()

### Task 5 — Visual Verification

Plots a sample of generated grid images with their YOLO bounding boxes overlaid, confirming that every face has a correct annotation.

In [ ]:
def annotate_ax(ax, img_path, label_path):
    img = Image.open(img_path)
    w, h = img.size
    ax.imshow(img)
    with open(label_path) as f:
        for line in f.read().strip().splitlines():
            parts = line.split()
            if not parts:
                continue
            ci = int(parts[0])
            cx, cy, bw, bh = (float(v) for v in parts[1:5])
            x1, y1 = (cx - bw / 2.0) * w, (cy - bh / 2.0) * h
            x2, y2 = (cx + bw / 2.0) * w, (cy + bh / 2.0) * h
            ax.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                            fill=False, edgecolor='red', lw=2))
            ax.text(x1, max(2, y1 - 8), CLASS_NAMES[ci], color='red',
                    fontsize=7, fontweight='bold')
    ax.axis('off')

n_show = min(4, len(manifest))
sample = random.sample(manifest, n_show)
fig, axes = plt.subplots(1, n_show, figsize=(6 * n_show, 6))
for ax, m in zip(np.atleast_1d(axes).flatten(), sample):
    annotate_ax(ax, m['image'], m['label'])
    ax.set_title(f"{m['n_faces']} faces", fontsize=10)
plt.suptitle("Grid composites with YOLO bounding boxes", fontsize=13)
plt.tight_layout()
plt.show()

### Task 6 — Train / Val / Test Split

The full set is split 70% train / 15% validation / 15% test with a fixed seed, so the split is reproducible and no image appears in more than one partition. Because every identity appears in every split (9 of 13 per image, drawn randomly), the split is identity-balanced by construction.

In [ ]:
# Reset split dirs
for split_name in ('train', 'val', 'test'):
    for kind in ('images', 'labels'):
        d = os.path.join(OUTPUT_DIR, kind, split_name)
        if os.path.isdir(d):
            shutil.rmtree(d)

# 70/15/15 split (reproducible)
train_list, val_test_list = train_test_split(
    manifest, test_size=(VAL_RATIO + TEST_RATIO), random_state=GLOBAL_SEED
)
val_list, test_list = train_test_split(
    val_test_list, test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO), random_state=GLOBAL_SEED
)

def split_identity_dist(items):
    counts = {n: 0 for n in CLASS_NAMES}
    for m in items:
        for c in m['classes']:
            counts[c] += 1
    return counts

for split_name, items in [('train', train_list), ('val', val_list), ('test', test_list)]:
    img_dir = os.path.join(OUTPUT_DIR, 'images', split_name)
    lbl_dir = os.path.join(OUTPUT_DIR, 'labels', split_name)
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)
    for m in items:
        shutil.copy2(m['image'], os.path.join(img_dir, os.path.basename(m['image'])))
        shutil.copy2(m['label'], os.path.join(lbl_dir, os.path.basename(m['label'])))
    dist = split_identity_dist(items)
    print(f"{split_name:>5s} : {len(items):>3d} images | identity-face counts: {dist}")

# Drop staging
if os.path.isdir(staging):
    shutil.rmtree(staging)

# Write data.yaml
yaml_lines = [
    "# Project 1 \u2014 Synthetic Multi-Celebrity Grid Detection Dataset",
    "# Auto-generated by Milestone 2 (celebrity_detection)",
    f"path: {OUTPUT_DIR}",
    "train: images/train",
    "val: images/val",
    "test: images/test",
    "",
    f"nc: {NUM_CLASSES}",
    "names:",
]
for i, name in enumerate(CLASS_NAMES):
    yaml_lines.append(f"  {i}: {name}")
yaml_content = "\n".join(yaml_lines) + "\n"
with open(os.path.join(OUTPUT_DIR, 'data.yaml'), 'w') as f:
    f.write(yaml_content)
print("\nWrote data.yaml:")
print(yaml_content)

## Milestone 2 — Deliverables Summary

- **Synthetic grid dataset**: `NUM_IMAGES` composite frames at 640×640, each a 3×3 grid of 9 distinct celebrity faces.
- **YOLO annotations**: every face has a tight, normalised bounding box (`<class> <cx> <cy> <w> <h>`).
- **No image reuse**: each source face crop appears in at most one grid composite.
- **Clean split**: reproducible 70 / 15 / 15 train/val/test, identity-balanced, no leakage.
- **`data.yaml`**: ready for `YOLO("yolov8n.pt").train(data=".../data.yaml", ...)` in Milestone 3.

The dataset lives in the repository under `data/synthetic_grid/`. Running this notebook (seed `42`) reproduces the full pipeline end-to-end.